# Build a Knowledge Graph for Your vPC / MLAG Design — the guided version

*A warm, step-by-step lab for data-center switching engineers. You bring the vPC. We bring the graph.*

You already run a tiny distributed system in the top of every rack. It is called a **vPC**
(Virtual Port Channel — Cisco's name; other vendors call it **MLAG**). Two switches agree,
over a **peer-link** and a **peer-keepalive**, to act as *one* switch to everything below them.
A server plugs a cable into each switch, bonds them into a single port-channel, and never
knows there are two boxes up there. That agreement is a graph: two peers, the links between
them, and every device hanging off them.

A **knowledge graph** is that same picture — except **you** choose the nodes. And that one
freedom lets you add the node no `show vpc` was ever designed to hold: the **business service**
that quietly loses its second path — or drops entirely — when a peer fails.

By the end of this notebook you will have built, from an empty page, a small graph that
answers the two questions a `show vpc` blurs together:

> *"The peer-link just dropped. Which services actually **lose reachability** — and which merely
> **lose their backup** and keep running?"*

and watched it name them separately — a distinction that lives in nobody's CLI output.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing in the dark. A knowledge graph is just
**structured facts** (things, and the named links between them) plus **queries** that walk those
links. Everything is **deterministic and inspectable** — run it twice, get the same answer, and
you can point at every node that produced it. If you can read `show vpc`, you can read every
line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python + networkx +
matplotlib** (both already installed in Colab). No database, no Docker, no credentials. Press
*Runtime -> Run all*, or run each cell in order with **Shift+Enter**.


## The whole vocabulary — five words, each one a vPC thing you already know

Before any code, here is the *entire* mental model. Everything after this is these five ideas,
repeated. You don't have to learn what any of them *mean* — only which vPC thing each maps onto.

| Graph word | Plain meaning | The vPC thing it already is |
|---|---|---|
| **node** | a thing | a peer switch, a downstream server, a service |
| **edge** | a named, directed link between two nodes | a peer-link, a keepalive, "this server dual-homes to that switch" |
| **label** | a node's *type* | `Switch`, `Device`, `Service` — like "is this a peer or a leaf?" |
| **property** | a fact stored *on* a node or edge | `role='primary'`, `state='down'`, `attach='orphan'` |
| **traversal** | walking edges to answer a question | "if this peer dies, what downstream is cut off?" |

That's it. Hold those five words and the rest is just your day job wearing a new hat.

One idea deserves a spotlight, because it is the whole trick: **a fact can live on an edge, not
just a node.** "The peer-link is down" is not really a property of switch A or switch B — it is a
property of the *link between them*. vPC already knows this in its bones (peer-link state, keepalive
state are about the *adjacency*, not the boxes). In a graph you write it down literally: the failure
lives **on the edge**. Watch for that moment — it is where a rack diagram turns into something you
can *query*.


## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in memory. We use
a **`DiGraph`** — a *directed* graph, where every edge has a direction, an arrow. That matters here:
`server -> switch` ("this server attaches to this switch") is a different statement from
`switch -> server`. vPC is full of directional truth — who is primary, which way the keepalive is
measured — so directed edges are the honest choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab — nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print("Empty graph ready:", G)
print("Nodes:", G.number_of_nodes(), " Edges:", G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That empty `G` is your
blank whiteboard. Every step from here just adds nodes and edges to it.


## Step 1 — The two peer switches (they agree to act as one)

**Theory.** A vPC starts with exactly **two** switches that join a shared **vPC domain** and take
**roles**: one is elected **primary**, the other **secondary**. Those roles are quiet most of the
time — but they decide *who wins* when the peer-link breaks, so they are the first fact worth
writing down. Everything downstream trusts that these two boxes behave as a single logical switch.

In graph terms, each peer becomes a **node** with a **label** of `Switch` and a couple of
**properties**: its `role` and the `vpc_domain` it belongs to. We'll model a classic top-of-rack
pair: **dc-sw-01** (primary) and **dc-sw-02** (secondary), both in vPC domain 10.

Notice we are *not* dumping the port-channel member list or the MAC table in here. A node per
switch — not a node per interface. We store the *shape* of the design, and we'll come back to that
principle by name later.


In [ ]:
# add_node(name, **properties). The name is a unique handle; role & vpc_domain are facts on it.
G.add_node("dc-sw-01", label="Switch", role="primary",   vpc_domain=10, status="up")
G.add_node("dc-sw-02", label="Switch", role="secondary", vpc_domain=10, status="up")

for n, d in G.nodes(data=True):
    if d.get("label") == "Switch":
        print(f'{n}: role={d["role"]}, vpc_domain={d["vpc_domain"]}, status={d["status"]}')

**Checkpoint.** Two switch nodes print: `dc-sw-01` is `primary`, `dc-sw-02` is `secondary`, both in
domain 10 and `up`. They're floating unconnected for now — the next step wires them together, and
that wire is the heart of a vPC.


## Step 2 — The peer-link and the keepalive (the fabric between the peers)

**Theory.** Two switches only *act as one* because of two links between them:

- the **peer-link** — a fat bundle that carries actual data between the peers plus the vPC control
  state (which MACs, which port-channels are synced). This is the load-bearing one.
- the **peer-keepalive** — a thin, separate heartbeat whose only job is to let each peer answer one
  question: *"is my partner still alive?"* It deliberately runs on a different path from the peer-link,
  because the peers need to tell apart "the peer-link died" from "my partner died" — and, as you'll
  see, they react to those two very differently.

So we add two **edges**, each with a `rel` and a `state` **property** hung right on it. Read
`dc-sw-01 --PEER_LINK(up)--> dc-sw-02` literally: *"these peers share a peer-link, and it is up."*
Both are healthy for now — we'll break one later, on purpose, and watch the graph react.


In [ ]:
# add_edge(source, target, **properties). The link state is a property ON the edge.
G.add_edge("dc-sw-01", "dc-sw-02", rel="PEER_LINK", state="up")
G.add_edge("dc-sw-02", "dc-sw-01", rel="KEEPALIVE", state="up")

for u, v, d in G.edges(data=True):
    print(f'{u} --{d["rel"]}({d["state"]})--> {v}')

**Checkpoint.** Two edges print — one `PEER_LINK` and one `KEEPALIVE`, both `up`. Those two links
are the entire reason the pair below the rack sees *one* switch instead of two. Keep an eye on the
peer-link: it's the edge whose `state` we'll flip to run the failure.


## Step 3 — Downstream devices: dual-homed vs orphan (this is the pivotal step)

**Theory.** Here is the idea the whole lab pivots on. Every device below the pair attaches in one of
**two ways**, and the difference is the entire story of what survives a failure:

- **dual-homed** — the device runs one cable to *each* peer and bonds them into a single vPC
  port-channel. It has **two paths**. This is the whole point of vPC: lose one peer and the device
  keeps running on the other, usually without dropping a packet.
- **orphan** — the device is plugged into **only one** peer (a single NIC, a device that can't bond,
  a miscabling). It has **one path**. It is a single point of failure hiding inside a redundant design,
  and it is exactly what a rack diagram never makes obvious.

We model that difference as the **edge type**. A dual-homed device gets **two** `DUAL_HOMED_TO`
edges — one to each peer. An orphan gets a single `ORPHAN_OF` edge to its one peer. The attach style
isn't a note in someone's head anymore; it's the shape of the edges.

The three devices we wire, chosen so every case shows up cleanly:

- **esxi-01** — a hypervisor, **dual-homed** to both peers (the healthy, redundant case).
- **backup-nas** — a storage box, **orphan of the secondary** (`dc-sw-02`).
- **mgmt-host** — a jump box, **orphan of the primary** (`dc-sw-01`).

Two orphans on *opposite* peers is deliberate — it's how we'll later prove the answer depends on
*which* peer fails, not just *that* one did.


In [ ]:
# dual-homed  = two edges, one to EACH peer.   orphan = one edge to its single peer.
G.add_node("esxi-01",    label="Device", attach="dual-homed")
G.add_node("backup-nas", label="Device", attach="orphan")
G.add_node("mgmt-host",  label="Device", attach="orphan")

# esxi-01 dual-homes to BOTH peers -> two edges
G.add_edge("esxi-01", "dc-sw-01", rel="DUAL_HOMED_TO")
G.add_edge("esxi-01", "dc-sw-02", rel="DUAL_HOMED_TO")
# backup-nas hangs off the SECONDARY only;  mgmt-host off the PRIMARY only
G.add_edge("backup-nas", "dc-sw-02", rel="ORPHAN_OF")
G.add_edge("mgmt-host",  "dc-sw-01", rel="ORPHAN_OF")

for dev in ("esxi-01", "backup-nas", "mgmt-host"):
    peers = [f'{d["rel"]} {v}' for _, v, d in G.out_edges(dev, data=True)]
    print(f'{dev:12} ({G.nodes[dev]["attach"]:10}) -> ' + ", ".join(peers))

**Checkpoint.** `esxi-01` prints with **two** `DUAL_HOMED_TO` edges (one per peer); `backup-nas` and
`mgmt-host` each have a **single** `ORPHAN_OF` edge, to opposite peers. The redundancy — and the two
hidden single-points-of-failure — are now literal facts in the graph, not tribal knowledge.


## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the picture.
We'll colour nodes by their **label** (switches one colour, devices another, services a third) and
draw any **down** link as a dashed red arrow so a failure jumps out. Same instinct as a rack diagram —
except this one is generated *from the data*, so it can never drift out of sync with the truth.


In [ ]:
def draw(G, title="vPC knowledge graph"):
    colors = {"Switch": "#3aa0ff", "Device": "#0f7f74", "Service": "#c8702a"}
    node_colors = []
    for n in G:
        d = G.nodes[n]
        if d.get("label") == "Switch" and d.get("status") == "down":
            node_colors.append("#d34b3f")          # a dead peer glows red
        else:
            node_colors.append(colors.get(d.get("label"), "#cccccc"))
    pos = nx.spring_layout(G, seed=7)               # fixed seed => stable, repeatable layout

    plt.figure(figsize=(10, 7))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1900, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=9)

    # Curve every edge slightly so the reciprocal PEER_LINK / KEEPALIVE arrows between the two
    # switches don't land on the same straight line and hide each other's label + arrowhead.
    curve = "arc3,rad=0.15"
    down  = [(u, v) for u, v, d in G.edges(data=True) if d.get("state") == "down"]
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=16,
                           edge_color="#8894a0", connectionstyle=curve)
    nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=16,
                           edge_color="#d34b3f", width=2.0, style="dashed", connectionstyle=curve)
    elabels = {(u, v): d.get("rel", "") for u, v, d in G.edges(data=True)}
    try:   # newer networkx (>=3.4) can curve the labels to match the curved edges
        nx.draw_networkx_edge_labels(G, pos, font_size=7, edge_labels=elabels, connectionstyle=curve)
    except TypeError:   # older networkx: draw straight labels (edges still curve, so arrows separate)
        nx.draw_networkx_edge_labels(G, pos, font_size=7, edge_labels=elabels)

    plt.axis("off"); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Reading the picture.** You should see the two blue switches joined by `PEER_LINK` and `KEEPALIVE`
arrows, `esxi-01` (teal) reaching up to *both* of them, and the two orphan boxes each touching just
one peer. No red yet — everything is healthy. This is still just a topology. It becomes a *knowledge*
graph in the next step, when we add the thing a rack diagram has never carried: the business behind
each box.


## Step 4 — The services: the node vPC never had

**Theory.** This is the node your switches were never designed to hold, and the reason the whole lab
exists. vPC knows peers, port-channels, MACs, member ports. It has never once known that `esxi-01`
hosts the **Order VM** where checkout runs, that `backup-nas` holds the **Backup Service**, or that
`mgmt-host` is the **Mgmt Console** you'd use to *fix* an outage.

So we add a **Service** node for each, with a `criticality` property, and wire it to its device with
a `RUNS_ON` edge: *"this service runs on that device."* One edge each — and now an attach-style fact
(dual-homed vs orphan) and a business fact are welded together in a place you can query. No `show`
command holds these links. They've never lived anywhere except a wiki nobody trusts. You're about to
make them first-class, walkable facts.


In [ ]:
G.add_node("Order VM",     label="Service", criticality="critical")
G.add_node("Backup Service", label="Service", criticality="high")
G.add_node("Mgmt Console",  label="Service", criticality="medium")

G.add_edge("Order VM",      "esxi-01",    rel="RUNS_ON")
G.add_edge("Backup Service", "backup-nas", rel="RUNS_ON")
G.add_edge("Mgmt Console",  "mgmt-host",  rel="RUNS_ON")

print("Graph now has", G.number_of_nodes(), "nodes and", G.number_of_edges(), "edges.")
for u, v, d in G.edges(data=True):
    if d.get("rel") == "RUNS_ON":
        print(f'  {u} -RUNS_ON-> {v}  (attach: {G.nodes[v]["attach"]})')
draw(G, title="vPC knowledge graph — now with the business services")

**Checkpoint.** Three orange `Service` nodes now hang off their devices, and the printout pairs each
service with its device's attach style — `Order VM` on a *dual-homed* box, `Backup Service` and
`Mgmt Console` on *orphans*. In the redrawn picture you can *trace with your finger*: a peer -> the
device attached to it -> the service on that device. Next we make the computer trace it, and we break
something first.


## Step 5 — Model the failure: the peer-link drops (keepalive still up)

**Theory.** Now the classic vPC failure, and the one everyone gets wrong at 3 AM. The **peer-link**
goes down, but the **keepalive** stays up. So each peer thinks: *"my partner is still alive (keepalive
is up), I just can't sync with it (peer-link is down)."* Two live switches, both claiming the same
port-channels, with no way to coordinate — that's a recipe for loops and duplicate frames.

vPC's rule for this: **the secondary suspends its vPC member ports.** The primary keeps forwarding;
the secondary stops presenting the dual-homed bundles. The consequences split cleanly, and this split
is the entire payoff of the lab:

- **dual-homed devices keep only the primary path** — they *lose redundancy* but *stay up*. `esxi-01`
  is now riding a single switch; one more failure and it's gone, but right now Order VM still serves.
- **orphan devices on the secondary are cut off** — `backup-nas` (orphan of `dc-sw-02`) *loses
  reachability*. Its Backup Service actually drops.
- **orphan devices on the primary are fine** — `mgmt-host` sits on the peer that stays in charge.

**(Important, so we're precise about vPC.)** NX-OS does **not** suspend orphan ports automatically. By
default only the secondary's **vPC member ports** go down; an orphan on the secondary keeps forwarding
through the secondary's *own* uplinks — unless you've configured **`vpc orphan-port suspend`**, or the
orphan's only return path happens to cross the now-dead peer-link. This graph models the **hardened,
recommended design** where `vpc orphan-port suspend` is set, so an orphan on the suspended peer really
does drop. If your rack doesn't set it, model those orphans as *lost redundancy* instead of *lost
reachability* — the graph makes that a one-line change.

We encode the whole incident as **one property on one edge**: the peer-link's `state`.


In [ ]:
# The entire incident is one property flip on one edge.
G["dc-sw-01"]["dc-sw-02"]["state"] = "down"     # peer-link DOWN
# (keepalive edge is left 'up' — that combination is what suspends the secondary's vPC ports)

for u, v, d in G.edges(data=True):
    if d.get("rel") in ("PEER_LINK", "KEEPALIVE"):
        print(f'{u} --{d["rel"]}({d["state"]})--> {v}')

**Checkpoint.** The `PEER_LINK` now reads `down` while the `KEEPALIVE` stays `up`. That exact
combination — dead peer-link, live heartbeat — is the trigger for the secondary to suspend its vPC
member ports. The incident is now a fact sitting in your graph, waiting to be queried.


## Step 6 — The 3 AM question: two blast radii, not one

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to answer a
question — and the whole reason a graph beats `show vpc` here is that it can answer **two different**
questions at once and *keep them separate*:

1. **Lost reachability** — services that actually **drop**. These are the orphans stranded by the
   failure. This is the pager-worthy list.
2. **Lost redundancy** — services that **keep running but have no backup left**. These are the
   dual-homed devices now riding a single peer. This is the *"fix it before the next failure"* list.

Our walk reads the current state of the graph and decides which failure mode it's in:

- if a whole **peer switch is down** -> its orphans lose reachability; dual-homed survive on the other
  peer but lose redundancy;
- if the **peer-link is down while the keepalive is up** -> the secondary suspends its vPC member
  ports, so dual-homed devices lose redundancy, and — *in a design that sets `vpc orphan-port
  suspend`, which is what we model here* — the **secondary's orphans lose reachability** too. (Recall
  the caveat from Step 5: without that opt-in setting, NX-OS leaves orphan ports forwarding, so those
  orphans would only lose redundancy. The graph would model that by moving them to the softer list.)

Every hop is one edge. And because the query is *conditioned on state*, fixing the link makes the
answer change — that's the difference between a static diagram and a live model. Run it.


In [ ]:
def vpc_state(G):
    """Pull the peers, their roles, and the peer-link / keepalive states out of the graph."""
    switches = {n: d for n, d in G.nodes(data=True) if d.get("label") == "Switch"}
    primary   = next(n for n, d in switches.items() if d.get("role") == "primary")
    secondary = next(n for n, d in switches.items() if d.get("role") == "secondary")
    def link_state(rel):
        return next((d.get("state") for _, _, d in G.edges(data=True) if d.get("rel") == rel), None)
    return primary, secondary, link_state("PEER_LINK"), link_state("KEEPALIVE")

def services_on(G, dev):
    return [s for s, _, d in G.in_edges(dev, data=True) if d.get("rel") == "RUNS_ON"]

def orphans_of(G, sw):
    return [dev for dev, s, d in G.edges(data=True) if d.get("rel") == "ORPHAN_OF" and s == sw]

def dual_homed(G):
    return sorted({dev for dev, _, d in G.edges(data=True) if d.get("rel") == "DUAL_HOMED_TO"})

def blast_radius(G):
    """Return two lists: services that LOSE REACHABILITY, and services that only LOSE REDUNDANCY."""
    primary, secondary, peer, keep = vpc_state(G)
    prim_up = G.nodes[primary].get("status", "up") == "up"
    sec_up  = G.nodes[secondary].get("status", "up") == "up"
    lost_reach, lost_redund = [], []

    if not (prim_up and sec_up):
        # a whole peer is down
        dead  = primary if not prim_up else secondary
        alive = secondary if not prim_up else primary
        for dev in orphans_of(G, dead):
            for svc in services_on(G, dev):
                lost_reach.append((svc, dev, f"orphan of {dead} (peer down)"))
        for dev in dual_homed(G):
            for svc in services_on(G, dev):
                lost_redund.append((svc, dev, f"survives on {alive}, lost {dead}"))
    elif peer == "down" and keep == "up":
        # peer-link down, keepalive up -> secondary suspends its vPC member ports
        for dev in orphans_of(G, secondary):
            for svc in services_on(G, dev):
                lost_reach.append((svc, dev, f"orphan of {secondary}; its vPC ports suspended"))
        for dev in dual_homed(G):
            for svc in services_on(G, dev):
                lost_redund.append((svc, dev, "secondary member ports suspended; primary path only"))
    # else: everything healthy -> both lists stay empty
    return lost_reach, lost_redund

reach, redund = blast_radius(G)
print("LOST REACHABILITY (service actually drops):")
for svc, dev, why in reach:   print(f'   DOWN  {svc:16} on {dev:12} — {why}')
print("\nLOST REDUNDANCY (still up, but no backup left):")
for svc, dev, why in redund:  print(f'   WARN  {svc:16} on {dev:12} — {why}')

The peer-link only ever told you a bundle went down — one line in a log. Your graph just told you
two *different* things that log blurs together: **Backup Service actually dropped** (its box was an
orphan on the suspended peer), while **Order VM is still serving but has lost its safety net**. And
notice `Mgmt Console` — the orphan on the *primary* — never appears: the graph correctly left the
healthy path alone. That reachable-vs-redundant split is the whole pitch of a knowledge graph, and
you just built it from an empty page.


## Step 7 — What-if: repair the peer-link, then break it again

**Theory.** Because the failure lives on an edge as a plain property, you can *change* it and re-ask —
running "what if this fails?" (or "what if I fix it?") on a **model**, safely, with no one paged and no
maintenance window. Flip the peer-link back to `up`: both blast radii clear. Flip it to `down` again:
Backup Service drops and Order VM's warning returns. This is the humble beginning of pre-incident
planning — you can ask the graph what *would* break before it does.


In [ ]:
def summary(G, tag):
    reach, redund = blast_radius(G)
    print(f'{tag:22} lost_reachability={sorted({s for s,_,_ in reach})}  '
          f'lost_redundancy={sorted({s for s,_,_ in redund})}')

G["dc-sw-01"]["dc-sw-02"]["state"] = "up"     # repair the peer-link
summary(G, "peer-link UP:")

G["dc-sw-01"]["dc-sw-02"]["state"] = "down"   # break it again
summary(G, "peer-link DOWN:")

**Checkpoint.** With the peer-link `up`, both lists are empty — nothing at risk. Break it again and
`Backup Service` returns to *lost_reachability* while `Order VM` returns to *lost_redundancy*. Same
graph, same query — only the state on one edge changed, and the answer *responded*. That is what
makes it a model rather than a drawing.


## Your turn #1 — a second service stranded on the same orphan

Real boxes rarely run just one thing. Suppose `backup-nas` — the orphan on the secondary — also runs
an **Archive Service**. Add it, wire it with a `RUNS_ON` edge, and re-run `blast_radius`. You should
now see **two** services in the *lost_reachability* list from the same peer-link failure — because one
edge of extra truth is all it takes.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want to check.


In [ ]:
# TODO: add an "Archive Service" Service node (criticality="medium"),
#       then a RUNS_ON edge from "Archive Service" to "backup-nas".

# G.add_node(...)
# G.add_edge(...)

reach, redund = blast_radius(G)
print("lost_reachability:", sorted({s for s, _, _ in reach}))
print("lost_redundancy: ", sorted({s for s, _, _ in redund}))

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("Archive Service", label="Service", criticality="medium")
G.add_edge("Archive Service", "backup-nas", rel="RUNS_ON")

reach, redund = blast_radius(G)
print("lost_reachability:", sorted({s for s, _, _ in reach}))
print("lost_redundancy: ", sorted({s for s, _, _ in redund}))
```

Now both `Backup Service` **and** `Archive Service` come back in *lost_reachability* — they share the
same orphaned box, so the same failure takes them both. The blast radius grew the instant you told the
graph one more true thing; you didn't touch the query at all.
</details>


In [ ]:
# (Solution, applied — so the rest of the notebook has both services in the graph.)
G.add_node("Archive Service", label="Service", criticality="medium")
G.add_edge("Archive Service", "backup-nas", rel="RUNS_ON")
reach, _ = blast_radius(G)
print("Services that now lose reachability:", sorted({s for s, _, _ in reach}))

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of decision,
and it's worth seeing the seam between them:

- *"A `Switch` has a `role`. A `Device` is either `DUAL_HOMED_TO` two peers or `ORPHAN_OF` one. A
  `Service` `RUNS_ON` a `Device`."* — these are the **rules**: which node types exist, which edges are
  allowed, what shape a valid fact takes. That is the **schema**. Its fancy name is an **ontology** —
  and here's the friendly definition: *an ontology is the design guide of your graph, the agreed
  vocabulary written down as structure.* You already trust a vPC design standard to say what a valid
  pair looks like; an ontology does the same job for your graph.

- *"This particular switch is `dc-sw-02`, it is the `secondary`, and its peer-link is `down`."* —
  these are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has a hostname, an IP, or a serial
number, it is data (an instance), never schema.** `Switch` is schema; `dc-sw-02` is data.
`DUAL_HOMED_TO` is schema; "esxi-01 dual-homes to dc-sw-01" is data. Keep the vocabulary small and
stable; let the instances be many. That single discipline is the difference between a graph you can
grow for years and a swamp you abandon in a month.


## A peek at the real thing (1/2) — the same pair in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built are exactly
what you'd build in a real graph database like **Neo4j**, whose query language is **Cypher**. Cypher
reads almost like the arrows we've been drawing — `(node)-[:EDGE]->(node)`.

Here is **Steps 1–3 (the pair and one device)** as Cypher. `MERGE` means "find this node or create it
if missing" (so re-running is safe); `SET` assigns properties. This is *reference only* — you don't
run it here, it just shows the same idea in the grown-up tool:

```cypher
MERGE (p:Switch {id: 'dc-sw-01'}) SET p.role = 'primary',   p.vpc_domain = 10;
MERGE (s:Switch {id: 'dc-sw-02'}) SET s.role = 'secondary', s.vpc_domain = 10;

// the failure, as a property on the relationship — same as our edge.
// MERGE the edge WITHOUT the property, then SET it — so re-running with a new
// state updates the one edge instead of creating a duplicate. (Inlining {state:...}
// inside MERGE would make each distinct state a *different* edge to match-or-create.)
MERGE (p)-[pl:PEER_LINK]->(s) SET pl.state = 'down';
MERGE (s)-[ka:KEEPALIVE]->(p) SET ka.state = 'up';

MERGE (e:Device {id: 'esxi-01'}) SET e.attach = 'dual-homed';
MERGE (e)-[:DUAL_HOMED_TO]->(p);
MERGE (e)-[:DUAL_HOMED_TO]->(s);
MERGE (nas:Device {id: 'backup-nas'}) SET nas.attach = 'orphan';
MERGE (nas)-[:ORPHAN_OF]->(s);
```

See it? `(p)-[pl:PEER_LINK]->(s) SET pl.state = 'down'` is the same statement as our
`G.add_edge("dc-sw-01", "dc-sw-02", rel="PEER_LINK", state="down")`. Same nodes, same edge, same
fact-on-the-edge — one just happens to live in a database that scales to millions of nodes.


## A peek at the real thing (2/2) — the two blast radii in Cypher

Our `blast_radius` walk is a Python function with a couple of branches. In Cypher, each half of it is a
few lines, because in a graph database you **draw the shape you're looking for** and let the engine
find every match — no manual loops.

**Lost reachability** — orphans on the secondary when the peer-link is down:

```cypher
MATCH (:Switch {role:'primary'})-[pl:PEER_LINK {state:'down'}]->(sec:Switch {role:'secondary'})
MATCH (dev:Device)-[:ORPHAN_OF]->(sec)
MATCH (svc:Service)-[:RUNS_ON]->(dev)
RETURN svc.id AS lost_reachability;
```

**Lost redundancy** — dual-homed devices now down to a single path:

```cypher
MATCH (:Switch {role:'primary'})-[pl:PEER_LINK {state:'down'}]->(:Switch {role:'secondary'})
MATCH (dev:Device)-[:DUAL_HOMED_TO]->(:Switch)
MATCH (svc:Service)-[:RUNS_ON]->(dev)
RETURN DISTINCT svc.id AS lost_redundancy;
```

Read the first line of each as a sentence: *"match the primary whose PEER_LINK to the secondary is
down."* That's the same condition you wrote in Python — the pattern you `MATCH` **is** the traversal.
Different engine; identical thinking. If you can read the Python above, you can read the Cypher — you
already speak this language.


## Your turn #2 — make the query respond to a *different* failure

So far the *secondary* was the unlucky peer: its orphan (`backup-nas`) dropped, while `mgmt-host` on
the primary sat safe. Let's prove the model follows reality by failing the **other** peer instead.

1. first **repair the peer-link** (`G["dc-sw-01"]["dc-sw-02"]["state"] = "up"`) so we start clean;
2. now fail the whole **primary** switch: `G.nodes["dc-sw-01"]["status"] = "down"`;
3. re-run `blast_radius`. This time the victim flips: **`Mgmt Console`** (orphan of the primary)
   loses reachability, `Order VM` loses redundancy but survives on the secondary, and `Backup
   Service` — orphan of the *secondary* — is now perfectly **safe**.

The exact same query, a different failure, the opposite answer. Fill in the `# TODO`s, then run.


In [ ]:
# TODO 1: repair the peer-link so we start from a clean state:
#   G["dc-sw-01"]["dc-sw-02"]["state"] = "up"

# TODO 2: fail the whole PRIMARY switch:
#   G.nodes["dc-sw-01"]["status"] = "down"

reach, redund = blast_radius(G)
print("lost_reachability:", sorted({s for s, _, _ in reach}))
print("lost_redundancy: ", sorted({s for s, _, _ in redund}))

<details><summary><b>Solution — click to reveal</b></summary>

```python
G["dc-sw-01"]["dc-sw-02"]["state"] = "up"      # heal the peer-link first
G.nodes["dc-sw-01"]["status"] = "down"          # now the whole PRIMARY dies

reach, redund = blast_radius(G)
print("lost_reachability:", sorted({s for s, _, _ in reach}))   # -> ['Mgmt Console']
print("lost_redundancy: ", sorted({s for s, _, _ in redund}))   # -> ['Order VM']
```

When the *secondary* was suspended, `backup-nas` was the casualty and `mgmt-host` was safe. Fail the
*primary* instead and it flips exactly: `Mgmt Console` drops, `Backup Service` is fine, and `Order VM`
rides out the failure on the secondary while flagged as no-longer-redundant. You never taught the query
about this scenario — you just told the graph the truth, and the traversal did the rest.
</details>


In [ ]:
# (Solution, applied — then we heal the primary so later cells start from a clean, healthy pair.)
G["dc-sw-01"]["dc-sw-02"]["state"] = "up"
G.nodes["dc-sw-01"]["status"] = "down"
reach, redund = blast_radius(G)
print("primary DOWN -> lost_reachability:", sorted({s for s, _, _ in reach}),
      "| lost_redundancy:", sorted({s for s, _, _ in redund}))

G.nodes["dc-sw-01"]["status"] = "up"          # restore the primary
reach, redund = blast_radius(G)
print("restored     -> lost_reachability:", sorted({s for s, _, _ in reach}),
      "| lost_redundancy:", sorted({s for s, _, _ in redund}))

## Make it yours — swap in *your* rack

Now the moment it becomes your tool, not mine. Change the values below to **one** dual-homed device
and **one** orphan *you* actually run, each with a service on it. We fail the peer-link for you, so
your services show up. Run it, and watch **your own service names** sort themselves into
*lost reachability* (your orphan) and *lost redundancy* (your dual-homed box) — proof that the machine
you built understands your rack, not just the demo's.

Keep it to the smallest slice that answers one real question. You can always add one more node
tomorrow — that's the whole promise of a graph you can grow.


In [ ]:
# --- change these to your rack ---
MY_DUAL_DEVICE = "my-esxi-02"      # a box cabled to BOTH peers
MY_DUAL_SERVICE = "Checkout API"    # what runs on it
MY_ORPHAN_DEVICE = "my-legacy-nas"  # a box cabled to ONE peer only
MY_ORPHAN_SERVICE = "Nightly Report"
# ---------------------------------

# your dual-homed box -> both peers
G.add_node(MY_DUAL_DEVICE, label="Device", attach="dual-homed")
G.add_edge(MY_DUAL_DEVICE, "dc-sw-01", rel="DUAL_HOMED_TO")
G.add_edge(MY_DUAL_DEVICE, "dc-sw-02", rel="DUAL_HOMED_TO")
G.add_node(MY_DUAL_SERVICE, label="Service", criticality="critical")
G.add_edge(MY_DUAL_SERVICE, MY_DUAL_DEVICE, rel="RUNS_ON")

# your orphan box -> the secondary only (so the peer-link failure strands it)
G.add_node(MY_ORPHAN_DEVICE, label="Device", attach="orphan")
G.add_edge(MY_ORPHAN_DEVICE, "dc-sw-02", rel="ORPHAN_OF")
G.add_node(MY_ORPHAN_SERVICE, label="Service", criticality="high")
G.add_edge(MY_ORPHAN_SERVICE, MY_ORPHAN_DEVICE, rel="RUNS_ON")

G["dc-sw-01"]["dc-sw-02"]["state"] = "down"   # your modelled failure
reach, redund = blast_radius(G)
print("If the peer-link fails:")
print("  LOSES REACHABILITY:", sorted({s for s, _, _ in reach}))
print("  LOSES REDUNDANCY: ", sorted({s for s, _, _ in redund}))

**Checkpoint.** Your own service names appear — the orphan's service (e.g. `Nightly Report`) in
*LOSES REACHABILITY*, and your dual-homed service (e.g. `Checkout API`) in *LOSES REDUNDANCY*,
alongside the demo's. That is the moment the tool stopped being a tutorial and became yours. Every
other box in your rack is just a few more lines away.


## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything. Don't. Here is the discipline, in one line:

> **Model the nouns of your design review, not the numbers of your monitoring dashboard.**

Peers, roles, peer-links, attach style, devices, services — the **nouns** you'd draw on a whiteboard
while arguing about a rack design — those belong in the graph. Interface counters, per-queue drops,
optical light levels, the full MAC table — those are the **numbers**; leave them in the systems that
already store them well, and let the graph *reference* them when it needs to. The graph holds the
*shape of the dependency*; your NMS holds the firehose. Keep that line sharp and your graph stays
queryable for years.

### Where to go next
- **The companion Neo4j lab** does this exact thing in a real graph database — same nodes, same edges,
  same two blast radii — so the Cypher peeks above become things you run.
- **Add the northbound.** Model each peer's uplinks to the spine as edges too, and "which services
  lose reachability if the whole pair is isolated?" becomes one more traversal.
- **Add the change layer.** Model a `ChangeEvent` node linked to what it touched, and "what changed
  right before this orphan appeared?" becomes a query instead of an archaeology dig.


## You built a knowledge graph

Look back at the distance. You started with an empty page and five plain words. You added two peer
switches and the links between them — a topology. Then you added how each device attaches (dual-homed
vs orphan) and the one node vPC never had, the **business service** — and the topology became
*knowledge*. Finally you asked it the question no `show vpc` answers cleanly, and it split the world
in two: the services that truly **drop**, and the ones that merely **lose their backup** — then
responded correctly every time you changed the failure underneath it.

The important idea was never vPC, and never networkx. It's this: **operational truth has a shape** —
a device dual-homes to two peers, a service runs on a device, a peer-link carries a state — and once
that shape is a graph, impact analysis stops being tribal knowledge and becomes a traversal. A
runbook tells you what happened last time. A graph you can query tells you what happens *this* time.

You already run two switches pretending to be one. Now you know how to build the graph that remembers
which services are trusting that illusion — and which ones will notice the moment it slips. Go model
one real orphan on your network, and ask it what breaks.
